# Занятие 1. Изображение как массив

**Курс «Введение в компьютерное зрение» · Innopolis University · Fall 2026**
Лекция-опора: **L1 «Изображение и видео как данные»** · **90 минут в классе** · ведёт ассистент

---

Сегодня собираем **контактный лист**: берём кадр, режем его на патчи, сортируем
патчи по средней яркости и раскладываем в сетку с подписями. Задача выглядит
игрушечной, но в ней ровно тот набор операций, на котором стоит весь курс:
срезы и ROI, порядок осей, типы данных, ресемплинг.

| Минуты | Что происходит |
|--------|----------------|
| 0–15 | Ассистент разбирает опорный пример |
| 15–65 | Вы делаете **TODO 1–3** |
| 65–80 | Разбор решения |
| 80–90 | **Мост к ДЗ 1 «Фотолаборатория»** |

Ничего сдавать не нужно: **зачёт ставится в классе** по факту работы (2 % итоговой оценки).
⭐ — необязательная звёздочка. Решение публикуется сразу после занятия.

## 0. Проверка окружения

Если ячейка ругается — зовите ассистента сразу, не тратьте время занятия.

In [ ]:
REPO_URL = "https://github.com/afanasyspb/iu-intro-cv.git"     # адрес репозитория курса (для Colab)

import sys, subprocess
try:
    import cvcourse
except ImportError:
    if "google.colab" in sys.modules:      # Colab: пакет курса ставится один раз за сессию
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "opencv-contrib-python==4.14.0.94", "git+" + REPO_URL], check=True)
        import cvcourse
    else:
        raise ImportError("пакет курса не установлен: из корня репозитория выполните "
                          "pip install -r requirements.txt   (docs/setup-guide.md)")

import cv2, numpy as np
from cvcourse import io as cio, viz, metrics

print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| Colab:", cvcourse.IN_COLAB)
assert cv2.__version__.startswith("4.14"), "курс собран на OpenCV 4.14.0.94"

## 1. Берём кадр

Три способа, в порядке предпочтения. Первый интереснее: на своём снимке
контактный лист получается осмысленным.

In [ ]:
def make_scene(w=640, h=440, seed=0):
    """Запасной вариант: синтетическая сцена, если нет ни фото, ни камеры."""
    img = np.full((h, w, 3), 205, np.uint8)
    rng = np.random.default_rng(seed)
    for i in range(14):
        c = tuple(int(v) for v in rng.integers(30, 240, 3))
        x, y = int(rng.integers(20, w - 140)), int(rng.integers(20, h - 140))
        if i % 3:
            cv2.rectangle(img, (x, y), (x + 110, y + 110), c, -1)
        else:
            cv2.circle(img, (x + 55, y + 55), 55, c, -1)
    return np.clip(img + rng.normal(0, 5, img.shape), 0, 255).astype(np.uint8)


IMG_PATH = "data/my_photo.jpg"            # 1) своё фото — положите сюда

img = None
if __import__("os").path.exists(IMG_PATH):
    img = cio.imread(IMG_PATH)
else:
    try:                                   # 2) один кадр с веб-камеры
        img = next(iter(cio.video_frames(0, max_frames=1)))
    except Exception as e:
        print("камера недоступна (%s) — берём синтетику" % type(e).__name__)

if img is None:
    img = make_scene()                     # 3) запасной вариант

img = cv2.resize(img, (640, 440), interpolation=cv2.INTER_AREA)
viz.show(img, "исходный кадр")

## 2. Опорный пример — разбирает ассистент

Здесь всё написано. Задача — **понять каждую строку**: дальше вы соберёте
то же самое, но сами.

In [ ]:
# ── опорный пример: один патч и его характеристики ───────────────────────────
y0, y1, x0, x1 = 100, 250, 180, 330
patch = img[y0:y1, x0:x1]                  # ROI — это СРЕЗ, а не копия

print("кадр  ", img.shape, img.dtype)
print("патч  ", patch.shape, "средняя яркость %.1f" % patch.mean())
print("patch.base is img:", patch.base is img)   # True — общая память!

vis = img.copy()
cv2.rectangle(vis, (x0, y0), (x1, y1), (17, 90, 197), 3)
viz.grid({"где вырезали": vis, "патч": patch}, cols=2, size=5)

---

## TODO 1 — характеристики массива *(≈ 10 минут)*

Напишите `describe(img)`, возвращающую словарь с полями
`h`, `w`, `channels`, `dtype`, `min`, `max`, `mean`.

Поле `channels` для одноканального изображения должно быть `1`, а не падать:
у чёрно-белого массива всего две оси.

> **Подсказка.** `img.shape` — кортеж длины 2 или 3. `img.mean()` возвращает
> `float`, а `img.min()` — тип NumPy; приведите к обычным `int` и `float`.

In [ ]:
def describe(image):
    """Пять характеристик массива изображения одним словарём."""
    # >>> SOL TODO 1 · 5–8 строк: посчитать поля и вернуть словарь
    h, w = image.shape[:2]
    channels = image.shape[2] if image.ndim == 3 else 1
    return {"h": int(h), "w": int(w), "channels": int(channels),
            "dtype": str(image.dtype),
            "min": int(image.min()), "max": int(image.max()),
            "mean": float(image.mean())}
    # <<< SOL


info = describe(img)
for k, v in info.items():
    print(f"{k:9s} {v}")

assert info["h"] == 440 and info["w"] == 640, "перепутаны высота и ширина"
assert info["channels"] == 3, "цветной кадр — три канала"
assert info["dtype"] == "uint8", "после imread ожидается uint8"
assert describe(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY))["channels"] == 1, (
    "для одноканального изображения channels должен быть 1")
print("TODO 1 ✔")

## TODO 2 — нарезать кадр на патчи *(≈ 15 минут)*

Напишите `cut_patches(image, rows, cols)` — список патчей, слева направо
и сверху вниз. Каждый патч — **независимая копия**, а не вид на исходный массив.

Проверить это просто: после `patches[0][:] = 0` исходный кадр меняться **не должен**.

> **Подсказка.** Границы считаются так, чтобы не потерять правый и нижний край
> при делении с остатком: `y0 = r * h // rows`, `y1 = (r + 1) * h // rows`.

In [ ]:
def cut_patches(image, rows=4, cols=6):
    """Режет кадр на rows × cols патчей. Каждый патч — независимая копия."""
    # >>> SOL TODO 2 · 6–10 строк: два цикла по рядам и столбцам; границы — целочисленным делением
    h, w = image.shape[:2]
    out = []
    for r in range(rows):
        for c in range(cols):
            y0, y1 = r * h // rows, (r + 1) * h // rows
            x0, x1 = c * w // cols, (c + 1) * w // cols
            out.append(image[y0:y1, x0:x1].copy())
    return out
    # <<< SOL


patches = cut_patches(img, 4, 6)
print("патчей:", len(patches), "| размер первого:", patches[0].shape)

before = img.copy()
patches[0][:] = 0
assert np.array_equal(img, before), (
    "патч оказался видом на исходный массив — обнулив патч, вы испортили кадр. "
    "Нужна копия")
assert len(patches) == 24, "ожидалось 4 × 6 = 24 патча"
assert sum(p.shape[0] for p in patches[::6]) == img.shape[0], (
    "патчи не покрывают кадр целиком — потерян край при делении с остатком")
patches = cut_patches(img, 4, 6)
print("TODO 2 ✔")
viz.grid(patches[:6], cols=6, size=2, suptitle="первые шесть патчей")

## TODO 3 — контактный лист *(≈ 20 минут)*

Напишите `contact_sheet(patches, cell=(96, 96), cols=6, interpolation=cv2.INTER_AREA)`:

1. отсортируйте патчи по **средней яркости**, от тёмных к светлым;
2. приведите каждый к размеру `cell` заданным флагом;
3. разложите в сетку по `cols` штук в ряд;
4. подпишите каждую ячейку её средней яркостью — `cv2.putText`.

> **Подсказка.** `sorted(patches, key=...)`. Готовый холст удобно создать через
> `np.zeros((rows*ch, cols*cw, 3), np.uint8)` и писать в него срезами.
> Не забудьте: `cv2.resize` принимает размер как `(ширина, высота)` — наоборот
> к `shape`.

In [ ]:
def contact_sheet(patches, cell=(96, 96), cols=6, interpolation=cv2.INTER_AREA):
    """Патчи, отсортированные по яркости, в сетке с подписями."""
    # >>> SOL TODO 3 · 8–12 строк: сортировка по яркости, холст np.full, resize каждого патча в ячейку, putText
    cw, ch = cell
    order = sorted(patches, key=lambda p: p.mean())
    rows = -(-len(order) // cols)
    sheet = np.full((rows * ch, cols * cw, 3), 30, np.uint8)
    for i, p in enumerate(order):
        tile = cv2.resize(p, (cw, ch), interpolation=interpolation)
        if tile.ndim == 2:
            tile = cv2.cvtColor(tile, cv2.COLOR_GRAY2BGR)
        r, c = divmod(i, cols)
        sheet[r * ch:(r + 1) * ch, c * cw:(c + 1) * cw] = tile
        cv2.putText(sheet, "%.0f" % p.mean(), (c * cw + 5, r * ch + ch - 7),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)
    return sheet
    # <<< SOL


sheet = contact_sheet(patches)
means = [p.mean() for p in sorted(patches, key=lambda p: p.mean())]

assert sheet.shape == (4 * 96, 6 * 96, 3), f"неожиданная форма листа: {sheet.shape}"
assert means == sorted(means), "патчи не отсортированы по яркости"
print("TODO 3 ✔  · самый тёмный %.1f, самый светлый %.1f" % (means[0], means[-1]))
viz.show(sheet, "контактный лист", size=7)

### Измеряем: с какого момента правило начинает работать

На лекции прозвучало: «для уменьшения берут `INTER_AREA`». Проверим — но не
на одном размере, а на нескольких. Уменьшаем патч до ячейки и возвращаем
обратно, считаем PSNR относительно оригинала.

In [ ]:
def roundtrip_psnr(patch, cell, flag):
    down = cv2.resize(patch, (cell, cell), interpolation=flag)
    back = cv2.resize(down, patch.shape[1::-1], interpolation=cv2.INTER_LINEAR)
    return metrics.psnr(patch, back)


FLAGS = [("NEAREST", cv2.INTER_NEAREST), ("LINEAR", cv2.INTER_LINEAR),
         ("AREA", cv2.INTER_AREA)]
table = {}
for cell in (96, 48, 24):
    k = patches[0].shape[1] / cell
    row = {n: float(np.mean([roundtrip_psnr(p, cell, f) for p in patches]))
           for n, f in FLAGS}
    table[cell] = row
    print("ячейка %3d (уменьшение %.1fx): " % (cell, k)
          + "  ".join("%s %.2f" % kv for kv in row.items())
          + "   -> " + max(row, key=row.get))

### Вопрос, на который отвечаем вслух

Посмотрите на таблицу: **при слабом уменьшении `INTER_AREA` не выигрывает.**
Он начинает выигрывать примерно с двукратного, и дальше отрыв растёт.

Почему так? И какой флаг вы возьмёте, если надо, наоборот, **увеличить** патч?

In [ ]:
# Проверяем только то, что устойчиво на любой картинке.
# Где именно проходит перелом, зависит от содержания кадра — это и надо увидеть
# в таблице выше, а не зашивать в ассерт.
strong = table[24]
assert max(strong, key=strong.get) == "AREA", (
    "при сильном уменьшении должен выигрывать INTER_AREA")
assert all(min(t, key=t.get) == "NEAREST" for t in table.values()), (
    "INTER_NEAREST не должен выигрывать ни на одном масштабе")
print("правило про AREA подтверждено — но только при достаточном уменьшении")

---

## ⭐ Звёздочка — контактный лист из видео

Если закончили раньше. Снимите 60 кадров с камеры (две-три секунды), возьмите
каждый пятый — получится 12 кадров; соберите из них контактный лист и измерьте
реальный FPS камеры: сколько кадров в секунду она отдала на самом деле.

Требует локального запуска: в Colab потокового видео нет.

In [ ]:
import time
frames, t0 = [], None
try:
    for f in cio.video_frames(0, max_frames=13, step=5, resize=(320, 220)):
        if t0 is None:                  # первый кадр не считаем: камера «включается» до секунды
            t0 = time.perf_counter()
            continue
        frames.append(f)
    dt = time.perf_counter() - t0
    print("кадров: %d | реальный FPS: %.1f" % (len(frames) * 5, len(frames) * 5 / dt))
    viz.show(contact_sheet(frames, cell=(128, 88), cols=4), "лист из видео", size=7)
except Exception as e:
    print("камера недоступна:", type(e).__name__, e)

---

## Мост к домашнему заданию

**ДЗ 1 «Фотолаборатория», дедлайн W6.**

| Сегодня | В ДЗ 1 |
|---------|--------|
| `describe` | вызывается как есть — отчёт по всей папке снимков |
| `cut_patches` | обобщается: патчи с перекрытием |
| `contact_sheet` | становится сеткой «до / после» для каждой операции |
| сравнение флагов по PSNR | тот же приём, но для гамма-коррекции и баланса белого |

Чего сегодня **не** делали и придётся сделать дома: работа с папкой файлов,
обработка ошибочных снимков и оформление отчёта.

Полное ТЗ — `homeworks/hw1-photolab/README.md`, выдаётся на W3.

## Зачёт за занятие

Ассистент ставит зачёт, если:

- [ ] ячейка проверки окружения прошла;
- [ ] **TODO 1–3 выполнены**, все ассерты проходят;
- [ ] вы можете объяснить, почему в `cut_patches` нужен `.copy()`;
- [ ] на вопрос про `INTER_AREA` есть внятный ответ.

**2 % итоговой оценки**, ставится в классе. Досылать ничего не нужно.